In [ ]:

import pyodbc
from ollama import chat


# ============================================================
# 1. SQL SERVER CONNECTION
# ============================================================

connection = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=DESKTOP-GC4P189;"
    "DATABASE=orderssystem;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = connection.cursor()


# ============================================================
# 2. PROFIT ANALYSIS AGENT
# ============================================================

def profit_analysis_agent(user_prompt):

    # --------------------------------------------------------
    # QUERY 1: OVERALL PROFIT SUMMARY
    # Assumption: one unit per order
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            COUNT(o.orderid) AS total_orders,

            COALESCE(SUM(o.amount), 0) AS total_revenue,

            COALESCE(SUM(p.costprice), 0) AS total_product_cost,

            COALESCE(
                SUM(o.amount - p.costprice), 0
            ) AS gross_profit,

            CAST(
                COALESCE(
                    SUM(o.amount - p.costprice) * 100.0 /
                    NULLIF(SUM(o.amount), 0),
                    0
                ) AS DECIMAL(10,2)
            ) AS gross_profit_margin

        FROM orders o

        INNER JOIN product p
            ON o.productid = p.productid
    """)

    r = cursor.fetchone()

    overall_data = f"""
    Total Orders: {r.total_orders}
    Total Revenue: {r.total_revenue}
    Total Product Cost: {r.total_product_cost}
    Gross Profit: {r.gross_profit}
    Gross Profit Margin Percentage: {r.gross_profit_margin}%
    """


    # --------------------------------------------------------
    # QUERY 2: PRODUCT-WISE PROFIT
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            p.productid,
            p.productname,
            p.category,

            COUNT(o.orderid) AS total_orders,

            COALESCE(SUM(o.amount), 0) AS total_revenue,

            COALESCE(
                SUM(p.costprice), 0
            ) AS total_product_cost,

            COALESCE(
                SUM(o.amount - p.costprice), 0
            ) AS gross_profit,

            CAST(
                COALESCE(
                    SUM(o.amount - p.costprice) * 100.0 /
                    NULLIF(SUM(o.amount), 0),
                    0
                ) AS DECIMAL(10,2)
            ) AS profit_margin

        FROM product p

        LEFT JOIN orders o
            ON p.productid = o.productid

        GROUP BY
            p.productid,
            p.productname,
            p.category

        ORDER BY gross_profit DESC
    """)

    rows = cursor.fetchall()

    product_data = "\n".join(
        f"Product ID: {r.productid}, "
        f"Product: {r.productname}, "
        f"Category: {r.category}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}, "
        f"Product Cost: {r.total_product_cost}, "
        f"Gross Profit: {r.gross_profit}, "
        f"Profit Margin: {r.profit_margin}%"
        for r in rows
    )


    # --------------------------------------------------------
    # QUERY 3: SALESMAN-WISE PROFIT
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            s.salesmanid,
            s.salesmanname,

            COUNT(o.orderid) AS total_orders,

            COALESCE(SUM(o.amount), 0) AS total_revenue,

            COALESCE(
                SUM(p.costprice), 0
            ) AS total_product_cost,

            COALESCE(
                SUM(o.amount - p.costprice), 0
            ) AS gross_profit,

            CAST(
                COALESCE(
                    SUM(o.amount - p.costprice) * 100.0 /
                    NULLIF(SUM(o.amount), 0),
                    0
                ) AS DECIMAL(10,2)
            ) AS profit_margin

        FROM salesman s

        LEFT JOIN orders o
            ON s.salesmanid = o.salesmanid

        LEFT JOIN product p
            ON o.productid = p.productid

        GROUP BY
            s.salesmanid,
            s.salesmanname

        ORDER BY gross_profit DESC
    """)

    rows = cursor.fetchall()

    salesman_data = "\n".join(
        f"Salesman ID: {r.salesmanid}, "
        f"Salesman: {r.salesmanname}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}, "
        f"Product Cost: {r.total_product_cost}, "
        f"Gross Profit: {r.gross_profit}, "
        f"Profit Margin: {r.profit_margin}%"
        for r in rows
    )


    # --------------------------------------------------------
    # QUERY 4: CUSTOMER-WISE PROFIT
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            c.custid,
            c.custname,

            COUNT(o.orderid) AS total_orders,

            COALESCE(SUM(o.amount), 0) AS total_revenue,

            COALESCE(
                SUM(p.costprice), 0
            ) AS total_product_cost,

            COALESCE(
                SUM(o.amount - p.costprice), 0
            ) AS gross_profit

        FROM customer c

        LEFT JOIN orders o
            ON c.custid = o.custid

        LEFT JOIN product p
            ON o.productid = p.productid

        GROUP BY
            c.custid,
            c.custname

        ORDER BY gross_profit DESC
    """)

    rows = cursor.fetchall()

    customer_data = "\n".join(
        f"Customer ID: {r.custid}, "
        f"Customer: {r.custname}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}, "
        f"Product Cost: {r.total_product_cost}, "
        f"Gross Profit: {r.gross_profit}"
        for r in rows
    )


    # --------------------------------------------------------
    # QUERY 5: MONTHLY PROFIT ANALYSIS
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            YEAR(o.dateofpurchase) AS profit_year,
            MONTH(o.dateofpurchase) AS profit_month,

            COUNT(o.orderid) AS total_orders,

            SUM(o.amount) AS total_revenue,

            SUM(p.costprice) AS total_product_cost,

            SUM(o.amount - p.costprice) AS gross_profit

        FROM orders o

        INNER JOIN product p
            ON o.productid = p.productid

        GROUP BY
            YEAR(o.dateofpurchase),
            MONTH(o.dateofpurchase)

        ORDER BY
            profit_year,
            profit_month
    """)

    rows = cursor.fetchall()

    monthly_data = "\n".join(
        f"Year: {r.profit_year}, "
        f"Month: {r.profit_month}, "
        f"Orders: {r.total_orders}, "
        f"Revenue: {r.total_revenue}, "
        f"Product Cost: {r.total_product_cost}, "
        f"Gross Profit: {r.gross_profit}"
        for r in rows
    )


    # --------------------------------------------------------
    # QUERY 6: PRODUCTS WITH NEGATIVE GROSS PROFIT
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            p.productid,
            p.productname,

            COUNT(o.orderid) AS total_orders,

            SUM(o.amount) AS total_revenue,

            SUM(p.costprice) AS total_product_cost,

            SUM(o.amount - p.costprice) AS gross_profit

        FROM orders o

        INNER JOIN product p
            ON o.productid = p.productid

        GROUP BY
            p.productid,
            p.productname

        HAVING SUM(o.amount - p.costprice) < 0

        ORDER BY gross_profit ASC
    """)

    rows = cursor.fetchall()

    if rows:
        loss_data = "\n".join(
            f"Product: {r.productname}, "
            f"Orders: {r.total_orders}, "
            f"Revenue: {r.total_revenue}, "
            f"Product Cost: {r.total_product_cost}, "
            f"Gross Loss: {r.gross_profit}"
            for r in rows
        )
    else:
        loss_data = (
            "No products with negative gross profit "
            "were found in the available order data."
        )


    # --------------------------------------------------------
    # QUERY 7: CHECK FOR MISSING PRODUCT COST INFORMATION
    # --------------------------------------------------------

    cursor.execute("""
        SELECT
            COUNT(*) AS orders_with_missing_product
        FROM orders o
        LEFT JOIN product p
            ON o.productid = p.productid
        WHERE p.productid IS NULL
           OR p.costprice IS NULL
    """)

    missing_data = cursor.fetchone()

    missing_product_count = missing_data.orders_with_missing_product


    # --------------------------------------------------------
    # 3. SEND RESULTS TO LOCAL OLLAMA MODEL
    # --------------------------------------------------------

    response = chat(
        model="llama3.2",
        messages=[
            {
                "role": "system",
                "content": """
You are an AI Profit Analysis Agent.

Analyse business profitability using only the SQL data provided.

IMPORTANT ACCOUNTING RULES:

1. Use only the supplied database figures.
2. Never invent customers, products, orders, costs, or profits.
3. The initial model assumes one unit per order.
4. Revenue is the selling amount recorded in orders.amount.
5. Product cost is product.costprice for each order.
6. Gross profit = Revenue - Product Cost.
7. Gross profit margin = Gross Profit / Revenue * 100.
8. Revenue is not profit.
9. Gross profit is not net profit.
10. Do not subtract expenses, salaries, rent, taxes, or other
    operating costs unless those figures are supplied.
11. If a product has negative gross profit, explain that its
    recorded selling amount is below its product cost.
12. If the previous revenue is zero, percentage comparisons
    cannot be calculated using the usual growth formula.
13. Clearly identify when data is missing or incomplete.
14. Never treat a missing or zero cost as proof that a product
    is free to produce or purchase.
15. Explain rankings using the actual supplied figures.
16. Distinguish total profit from profit margin.
17. Recommend actions only when supported by the data.
18. If the data is insufficient to answer a question,
    state the limitation clearly.
19. All monetary figures must retain their supplied values.
20. Remember that product cost and selling amount may need
    quantity, discounts, returns, and tax adjustments in a
    production accounting system.
"""
            },

            {
                "role": "user",
                "content": f"""
OVERALL PROFIT SUMMARY:
{overall_data}

PRODUCT-WISE PROFIT:
{product_data}

SALESMAN-WISE PROFIT:
{salesman_data}

CUSTOMER-WISE PROFIT:
{customer_data}

MONTHLY PROFIT:
{monthly_data}

PRODUCTS WITH NEGATIVE GROSS PROFIT:
{loss_data}

ORDERS WITH MISSING PRODUCT OR COST INFORMATION:
{missing_product_count}

USER QUESTION:
{user_prompt}

Analyse the available data and provide a clear,
structured, accurate answer.
"""
            }
        ]
    )

    return response.message.content


# ============================================================
# 4. USER INTERFACE
# ============================================================

print("\n==============================================")
print("          AI PROFIT ANALYSIS AGENT")
print("==============================================")

print("\nYou can ask ANY profit-related question.")

print("\nExamples:")
print("  What is the total gross profit?")
print("  Which product generates the highest profit?")
print("  Which products are making a loss?")
print("  Compare revenue and profit for all products.")
print("  Which salesman generated the highest profit?")
print("  Which customer generated the highest profit?")
print("  What is the gross profit margin?")
print("  Show monthly profit trends.")
print("  Which product has the lowest profit margin?")
print("  Give me a complete profit analysis report.")

print("\nNote: The initial calculation assumes one unit per order.")
print("Type 'exit' to quit.")


# ============================================================
# 5. ACCEPT ARBITRARY USER QUESTIONS
# ============================================================

while True:

    user_prompt = input(
        "\nEnter your profit analysis question: "
    ).strip()

    if user_prompt.lower() == "exit":
        print("\nExiting Profit Analysis Agent...")
        break

    if not user_prompt:
        print("Please enter a profit-related question.")
        continue

    try:
        print("\nAnalysing profit data...")

        result = profit_analysis_agent(user_prompt)

        print("\n==============================================")
        print("             PROFIT ANALYSIS")
        print("==============================================")
        print(result)
        print("==============================================")

    except Exception as e:
        print("\nERROR:")
        print("----------------------------------------------")
        print(e)


# ============================================================
# 6. CLOSE DATABASE CONNECTION
# ============================================================

cursor.close()
connection.close()

print("\nDatabase connection closed.")
print("Profit Analysis Agent stopped.")



          AI PROFIT ANALYSIS AGENT

You can ask ANY profit-related question.

Examples:
  What is the total gross profit?
  Which product generates the highest profit?
  Which products are making a loss?
  Compare revenue and profit for all products.
  Which salesman generated the highest profit?
  Which customer generated the highest profit?
  What is the gross profit margin?
  Show monthly profit trends.
  Which product has the lowest profit margin?
  Give me a complete profit analysis report.

Note: The initial calculation assumes one unit per order.
Type 'exit' to quit.



Enter your profit analysis question:  what is total gross profit no explanation



Analysing profit data...

             PROFIT ANALYSIS
**Total Gross Profit:**
$151,400.00



Enter your profit analysis question:  Which products are making a loss?



Analysing profit data...
